# IND320 Project Work, Part 4 - Machine Learning

GitHub: [IND320-kahaland](https://github.com/kahalandkh/IND320-kahaland) \
Streamlit: [IND320 Energy Project](https://ind320-kahaland-vqqy43wgawwsxmytoyb9ah.streamlit.app)

## 0. Setup and imports

In [ ]:
from datetime import datetime, timezone
import inspect
from itertools import product
import os
from pathlib import Path
import sys
from time import perf_counter

# Set the working directory to the project root before importing Streamlit
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
sys.path.insert(0, str(project_root))

import certifi
from dotenv import load_dotenv
import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
from pymongo import MongoClient

from streamlit.logger import set_log_level

set_log_level("ERROR")

load_dotenv()

from modules.entsoe_mongodb import load_weekly_cross_border
from modules.forecasting import combine_weekly_data, forecast_reservoir, evaluate_forecast
from modules.magasinstatistikk_data import load_magasinstatistikk_data
from modules.nb_exchange_rates import get_nb_exchange_rates, load_weekly_exchange_rates
from modules.weather import load_regional_hourly_weather, hourly_weather_to_weekly

## 1. Norges Bank Currency Exchange Rates

The `get_nb_exchange_rates()` function in `modules/nb_exchange_rates.py` downloads exchange rates from the [Norges Bank API](https://www.norges-bank.no/en/topics/Statistics/open-data/guide-data-warehouse/). It supports selecting currencies, date ranges and frequency, and returns a pandas DataFrame with the original metadata.

The example below downloads EUR/NOK and USD/NOK rates.

In [2]:
print(inspect.getsource(get_nb_exchange_rates))

@st.cache_data(ttl="24h")
def get_nb_exchange_rates(start_date, end_date, currencies=("EUR", "USD"), frequency="B"):
    """Download exchange rates against NOK from the Norges Bank API."""

    # Accept both a single currency ("EUR") and multiple currencies ("EUR", "USD")
    if isinstance(currencies, str):
        currencies = (currencies,)

    # Norges Bank uses '+' to separate currencies in the API URL
    currency_codes = "+".join(currencies)
    url = f"https://data.norges-bank.no/api/data/EXR/{frequency}.{currency_codes}.NOK.SP"

    params = {"startPeriod": str(start_date),
              "endPeriod": str(end_date),
              "format": "csv",
              "locale": "en"}

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    # Read the semicolon-separated CSV response directly into memory and preserve the API metadata
    df = pd.read_csv(StringIO(response.text), sep=";")

    if df.empty:
        raise ValueError("No exchange rate

In [3]:
rates = get_nb_exchange_rates("2025-12-22", "2025-12-31")
rates

,FREQ,Frequency,BASE_CUR,Base Currency,QUOTE_CUR,Quote Currency,TENOR,Tenor,DECIMALS,CALCULATED,UNIT_MULT,Unit Multiplier,COLLECTION,Collection Indicator,TIME_PERIOD,OBS_VALUE
0,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-22,10.1213
1,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-23,10.0535
2,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-29,10.0518
3,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-30,10.0587
4,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-31,10.0791
5,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-22,11.8875
6,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-23,11.8490
7,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-29,11.8270
8,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-30,11.8260
9,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-31,11.8430


The function returned exchange rates for both EUR and USD. There are no observations for 24 to 28 December, which covers the Christmas holidays and the weekend.

## 2. Timezone Investigation

I looked at how dates and timezones are handled in Magasinstatistikk, ENTSO-E, Open-Meteo and Norges Bank. I also checked how often observations are recorded and whether weekends and holidays affect data availability, since these differences affect how the datasets can be combined for forecasting.

### 2.1 Norges Bank

The API returns exchange rates with observation dates but no timezone aware timestamps. The metadata specifies a reference time of 14:15 CET, and this label also appears during summer time. According to the [Norges Bank FAQ](https://www.norges-bank.no/en/topics/statistics/exchange_rates/valutakursar-faq), daily rates are published around 16:00 CET on business days.

I used Business frequency, so there are gaps for weekends and other days without published rates. I kept the original dates and metadata without applying any timezone conversion. The gaps between observations need to be considered when combining exchange rates with the other datasets.

In [4]:
winter = get_nb_exchange_rates("2026-01-15", "2026-01-15", currencies="EUR")
summer = get_nb_exchange_rates("2026-06-01", "2026-06-01", currencies="EUR")

samples = pd.concat([winter, summer])
samples[["TIME_PERIOD", "BASE_CUR", "Collection Indicator"]]

,TIME_PERIOD,BASE_CUR,Collection Indicator
0,2026-01-15,EUR,ECB concertation time 14:15 CET
0,2026-06-01,EUR,ECB concertation time 14:15 CET


### 2.2 Magasinstatistikk

Magasinstatistikk provides weekly reservoir filling data, with `date_id` referring to Sundays, including those that fall on holidays such as Christmas. According to [NVE's documentation](https://www.nve.no/energi/analyser-og-statistikk/om-magasinstatistikken/), the weekly values are based on measurements from the end of Sunday or up to two days earlier. The data is normally published on Wednesdays at 13:00, but publication may be delayed around holidays.

The API does not include timezone offsets. The `next_publication_date` field shows 13:00 in both winter and summer, suggesting that the publication time follows Norwegian local time (CET/CEST), although this is not explicitly stated. I kept the weekly dates unchanged, since they are reference dates rather than exact measurement timestamps.

In [5]:
df_mag = load_magasinstatistikk_data()

# Compare winter, summer and Christmas observations
dates = pd.to_datetime(["2022-12-25", "2023-12-24", "2026-01-18", "2026-07-05"])
cols = ["date_id", "iso_year", "iso_week", "next_publication_date"]

df_mag.loc[df_mag["date_id"].isin(dates), cols].drop_duplicates().sort_values("date_id")

,date_id,iso_year,iso_week,next_publication_date
171,2022-12-25,2022,51,2023-01-04T13:00:00
390,2023-12-24,2023,51,2024-01-03T13:00:00
3444,2026-01-18,2026,3,2026-01-28T13:00:00
6364,2026-07-05,2026,27,2026-07-15T13:00:00


### 2.3 ENTSO-E

According to [ENTSO-E's documentation](https://transparencyplatform.zendesk.com/hc/en-us/articles/12786431986964-Response-Time-Zone), API responses always use UTC timestamps. In Part 2, I downloaded cross-border electricity flow data for 2024 and 2025, processed it using Spark and Cassandra, and stored the results in MongoDB. I had some problems with duplicate timestamps around daylight-saving changes when transferring the data from pandas to Spark, which I solved by explicitly preserving the UTC offset. The MongoDB data is not updated automatically, so comparisons with other datasets are limited to the dates available in the collection.

I used the existing MongoDB data to check the transition from summer to winter time on 26 October 2025. After interpreting the stored timestamps as UTC and converting them to `Europe/Oslo`, the hour between 02:00 and 03:00 appears twice, once with UTC+2 and once with UTC+1. The UTC timestamps are different, so the two occurrences can still be distinguished.

Unlike Norges Bank, ENTSO-E also reports data for weekends and holidays. The data includes both hourly and 15-minute observations. I already accounted for these different intervals when calculating transferred energy in Part 2. The different observation intervals and UTC timestamps need to be accounted for when combining this data with the weekly reservoir values.

In [6]:
client = MongoClient(os.environ["MONGODB_URI"], tlsCAFile=certifi.where())
collection = client["ind320_project"]["cross_border_flows"]

# Check timestamps around the October 2025 daylight saving transition
query = {"norway_area": "BZN|NO2",
         "foreign_area": "BZN|DK1",
         "direction": "in",
         "timestamp": {"$gte": datetime(2025, 10, 26, 0, tzinfo=timezone.utc),
                       "$lt": datetime(2025, 10, 26, 2, tzinfo=timezone.utc)}}

records = list(collection.find(query, {"_id": 0, "timestamp": 1}).sort("timestamp"))

timestamps = pd.to_datetime([row["timestamp"] for row in records], utc=True)

display(pd.DataFrame({"UTC": timestamps, "Norwegian time": timestamps.tz_convert("Europe/Oslo")}))

# Check whether stored observations are present during Christmas
for day in ["2025-12-24", "2025-12-25", "2025-12-26"]:
    start = pd.Timestamp(day, tz="Europe/Oslo").tz_convert("UTC")
    end = (pd.Timestamp(day, tz="Europe/Oslo") + pd.Timedelta(days=1)).tz_convert("UTC")

    count = collection.count_documents({"norway_area": "BZN|NO2",
                                        "foreign_area": "BZN|DK1",
                                        "direction": "in",
                                        "timestamp": {"$gte": start.to_pydatetime(), "$lt": end.to_pydatetime()}})
    print(f"{day}: {count} observations")
    
client.close()

,UTC,Norwegian time
0,2025-10-26 00:00:00+00:00,2025-10-26 02:00:00+02:00
1,2025-10-26 00:15:00+00:00,2025-10-26 02:15:00+02:00
2,2025-10-26 00:30:00+00:00,2025-10-26 02:30:00+02:00
3,2025-10-26 00:45:00+00:00,2025-10-26 02:45:00+02:00
4,2025-10-26 01:00:00+00:00,2025-10-26 02:00:00+01:00
5,2025-10-26 01:15:00+00:00,2025-10-26 02:15:00+01:00
6,2025-10-26 01:30:00+00:00,2025-10-26 02:30:00+01:00
7,2025-10-26 01:45:00+00:00,2025-10-26 02:45:00+01:00


2025-12-24: 96 observations
2025-12-25: 96 observations
2025-12-26: 96 observations


### 2.4 Open-Meteo

In Part 3, I used the [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) to download hourly ERA5 temperature and precipitation data from sampled locations within each region. I selected `Europe/Berlin` so that the requested dates follow local time rather than UTC. Berlin and Norway both use CET in winter and CEST in summer. As recommended in the API documentation, I used Unix timestamps to handle daylight saving changes. These represent seconds since 1 January 1970 UTC. In `regional_hourly_averages()`, I interpret these timestamps as UTC and then convert them to local time using pandas. This ensures that each observation gets the correct offset, including during daylight saving changes.

The example below uses data from 25-27 October 2025 for one location in NO3 to show how the daylight saving change is handled. On 26 October, there were 25 hourly observations, including two occurrences of 02:00, one with UTC+2 and the other with UTC+1. Open-Meteo also uses a fixed UTC offset when determining the requested date boundaries, which can leave out hours near midnight. In Part 3, I handled this by requesting one additional day before and after the selected dates, converting the timestamps to local time and then filtering to the dates the user selected.

The hourly data includes weekends and holidays. For longer periods, daily data could be an alternative if I only need weekly weather values. This would reduce the amount of data downloaded, although Open-Meteo's API limits would still need to be considered.

According to the Open-Meteo documentation, ERA5 data are updated daily with a delay of approximately five days. This means that the latest week in Magasinstatistikk may not yet have corresponding weather data. I therefore need to account for this delay when displaying both datasets in Streamlit.

In [7]:
# Look at one location to investigate timestamps
weather = load_regional_hourly_weather("NO3", start_date="2025-10-25", end_date="2025-10-27", n_points=1)

# Inspect the switch from summer to winter time
oct_26 = weather.loc[weather["date"].dt.strftime("%Y-%m-%d") == "2025-10-26"]

print(f"Observations on 26 October 2025: {len(oct_26)}")
display(oct_26.loc[oct_26["date"].dt.hour == 2, ["date"]])

# Check hourly observations during holidays
christmas_weather = load_regional_hourly_weather("NO3", start_date="2025-12-23", end_date="2025-12-27", n_points=1)

counts = christmas_weather.groupby(christmas_weather["date"].dt.strftime("%Y-%m-%d")).size()
print(counts.loc[["2025-12-24", "2025-12-25", "2025-12-26"]])

Observations on 26 October 2025: 25


,date
25,2025-10-26 02:00:00+02:00
26,2025-10-26 02:00:00+01:00


date
2025-12-24    24
2025-12-25    24
2025-12-26    24
dtype: int64


## 3. Reservoir Filling Forecasting

In this section, I will use River to forecast weekly reservoir filling levels. I will compare different SNARIMAX settings, training lengths and forecast horizons, and investigate whether external variables such as weather, exchange rates and electricity transfers improve the predictions.

I will evaluate the models using historical forecasts and use the results to choose default settings for Streamlit. The final forecasting function will also support predictions for future weeks.

### 3.1 Preparing weekly data

Before forecasting, I added functions to prepare the datasets at a weekly resolution, reusing the existing data loading functions where possible.

Magasinstatistikk already provides weekly reservoir filling values. I use `reservoir_fill_ratio` as the variable to predict and select electricity area NO3 for the experiments in this notebook. For exchange rates, I calculate the mean of the available business day observations. Weather data is aggregated into weekly mean temperature and total precipitation, while electricity imports and exports are summed across the international connections for the selected Norwegian bidding zone. I also calculate net imports as imports minus exports.

I use Norwegian Monday–Sunday weeks, labelled by Sunday, to align the datasets. I chose weekly resolution because Magasinstatistikk already provides weekly observations. Using daily or hourly resolution would not give me any additional reservoir observations, while monthly aggregation would hide changes within each month.

In [8]:
# Select the region for forecasting
region = "NO3"
area_number = int(region[2:])

# Reservoir filling for the selected electricity area
reservoir_weekly = (df_mag.loc[(df_mag["area_type"] == "EL") & (df_mag["area_number"] == area_number),
                               ["date_id", "reservoir_fill_ratio"]].set_index("date_id").sort_index())

# Weekly EUR/NOK and USD/NOK exchange rates
currency_weekly = load_weekly_exchange_rates(reservoir_weekly.index.min().strftime("%Y-%m-%d"),
                                             reservoir_weekly.index.max().strftime("%Y-%m-%d"))

# Weekly international electricity transfers
electricity_weekly = load_weekly_cross_border(f"BZN|{region}")

start_date = pd.Timestamp("2021-01-01")
last_sunday = reservoir_weekly["reservoir_fill_ratio"].last_valid_index()
end_date = last_sunday + pd.Timedelta(days=1) # include Monday 00:00 to capture Sunday's final precipitation hour

hourly_blocks = []

# Load weather in yearly blocks
for year in range(start_date.year, end_date.year + 1):
    block_start = max(start_date, pd.Timestamp(year=year, month=1, day=1))
    block_end = min(end_date, pd.Timestamp(year=year, month=12, day=31))
    hourly = load_regional_hourly_weather(region, block_start.strftime("%Y-%m-%d"), block_end.strftime("%Y-%m-%d"),
                                          n_points=100, batch_size=100)
    local_dates = hourly["date"].dt.tz_convert("Europe/Oslo").dt.tz_localize(None)
    hourly = hourly.loc[(local_dates >= block_start) & (local_dates < block_end + pd.Timedelta(days=1))]
    hourly_blocks.append(hourly)

# Convert hourly weather to complete weekly observations
hourly_weather = pd.concat(hourly_blocks, ignore_index=True).sort_values("date")
weather_weekly = hourly_weather_to_weekly(hourly_weather).loc[:last_sunday]

# Combine the selected weekly datasets
weekly_data = combine_weekly_data(reservoir_weekly, sources={"currency": currency_weekly,
                                                             "weather": weather_weekly,
                                                             "electricity": electricity_weekly})

# Show the available historical coverage
coverage = pd.DataFrame({"First week": weekly_data.apply(lambda s: s.first_valid_index()),
                         "Last week": weekly_data.apply(lambda s: s.last_valid_index()),
                         "Available weeks": weekly_data.notna().sum()})
display(coverage)
display(weekly_data.tail())

,First week,Last week,Available weeks
reservoir_fill_ratio,1995-01-08,2026-11-15,1663
eur_nok,1999-01-10,2026-11-15,1454
usd_nok,1995-01-15,2026-11-15,1662
temperature_mean_c,2021-01-10,2026-11-15,306
precipitation_total_mm,2021-01-10,2026-11-15,306
imports_mwh,2024-01-07,2025-12-28,104
exports_mwh,2024-01-07,2025-12-28,104
net_imports_mwh,2024-01-07,2025-12-28,104


,reservoir_fill_ratio,eur_nok,usd_nok,temperature_mean_c,precipitation_total_mm,imports_mwh,exports_mwh,net_imports_mwh
date_id,,,,,,,,
2026-10-18,0.724940,10.6487,9.50360,6.706483,41.940,NaN,NaN,NaN
2026-10-25,0.717647,10.5093,9.37124,7.529061,11.063,NaN,NaN,NaN
2026-11-01,0.720149,10.3668,9.29372,7.873263,38.837,NaN,NaN,NaN
2026-11-08,0.718705,10.4111,9.28732,5.613929,15.558,NaN,NaN,NaN
2026-11-15,0.697278,10.5790,9.50230,1.988682,42.612,NaN,NaN,NaN


The datasets cover quite different periods. Reservoir filling and USD/NOK go back to 1995, while EUR/NOK starts in 1999. I limited the weather data to 2021 onwards to avoid downloading unnecessarily large amounts of hourly data from the API, and the electricity data I collected cover 2024–2025.

### 3.2 Forecasting models and evaluation

I implemented the forecasting function in `modules/forecasting.py` so that I can reuse it in Streamlit. The function uses River's SNARIMAX model and allows me to change the model parameters, training length, forecast horizon and exogenous variables.

In [9]:
print(inspect.getsource(forecast_reservoir))

def forecast_reservoir(data, params, training_weeks=780, horizon=4, cutoff=None,
                       columns=None, exog_lag=None, use_calendar=True):
    """Forecast weekly reservoir filling using River's SNARIMAX model."""
    target = "reservoir_fill_ratio"

    if not isinstance(data, pd.DataFrame) or target not in data.columns:
        raise ValueError(f"Data must contain {target!r}.")
    if not isinstance(data.index, pd.DatetimeIndex) or data.index.hasnans or data.index.has_duplicates:
        raise ValueError("Data must have unique, valid Sunday dates as its index.")
    if data.index.tz is not None or not (data.index.weekday == 6).all() or not (data.index == data.index.normalize()).all():
        raise ValueError("Data must use timezone-naive Sunday date labels.")
    if not isinstance(training_weeks, (int, np.integer)) or isinstance(training_weeks, (bool, np.bool_)) or training_weeks < 1:
        raise ValueError("training_weeks must be a positive integer.")
    if not isin

#### 3.2.1 Grid search setup

I compare different SNARIMAX parameters, training periods of different lengths, horizons and exogenous variables. Since the datasets cover different time periods, I divide the experiments into three groups with different training lengths. Within each group, I use the same forecast dates so that the results can be compared directly. I use the reservoir observation dates as forecast cutoffs, rather than the publication dates, to keep all datasets aligned on a weekly basis. I also use lagged values for the exogenous variables, since their future values would not be known when making a forecast. I set the lag equal to the horizon.

In [10]:
currency_options = {"USD/NOK": ["usd_nok"],
                    "EUR/NOK": ["eur_nok"],
                    "USD + EUR": ["usd_nok", "eur_nok"]}

weather_options = {"Temperature": ["temperature_mean_c"],
                   "Precipitation": ["precipitation_total_mm"],
                   "Temperature + precipitation": ["temperature_mean_c", "precipitation_total_mm"]}

electricity_options = {"Imports": ["imports_mwh"],
                       "Exports": ["exports_mwh"],
                       "Net imports": ["net_imports_mwh"],
                       "Imports + exports": ["imports_mwh", "exports_mwh"]}

# Group experiments by training length and available forecast dates
search_groups = {"long_history": {"cutoffs": pd.date_range("2021-01-03", "2025-12-28", freq="13W-SUN"),
                                  "training_lengths": [520, 780],
                                  "options": {"Reservoir only": [], **currency_options}},
                 "medium_history": {"cutoffs": pd.to_datetime(["2025-03-30", "2025-06-29",
                                   "2025-09-28", "2025-12-28"]),
                                    "training_lengths": [104, 208],
                                    "options": {"Reservoir only": [], **currency_options, **weather_options}},
                 "short_history": {"cutoffs": pd.to_datetime(["2025-09-28", "2025-10-26", "2025-11-23", "2025-12-28"]),
                                   "training_lengths": [78], "options": {"Reservoir only": [], **currency_options, **weather_options, **electricity_options}}}

horizons = [4, 8, 12]
season_lengths = [52]  # Annual cycle in weekly data

orders = [(p, d, q, m, sp, sd, sq) for p, d, q, sp, sd, sq in product([0, 1, 2, 4], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1])
          for m in ([1] if not (sp or sd or sq) else season_lengths)]

candidate_count = sum(len(horizons) * len(g["training_lengths"]) * len(g["options"]) * 2 * len(orders) for g in search_groups.values())
print(f"Search grid: {candidate_count} candidate configurations")

Search grid: 25344 candidate configurations


Before running the grid search, I check for missing data in the selected training and evaluation periods, also taking the lag of the exogenous variables into account.

In [11]:
def has_complete_data(data, cutoffs, training_weeks, horizon, exog_columns):
    """Check that the required observations are available for all cutoffs."""

    for cutoff in cutoffs:
        train_dates = pd.date_range(end=cutoff, periods=training_weeks, freq="W-SUN")
        future_dates = pd.date_range(start=cutoff + pd.Timedelta(weeks=1),
                                     periods=horizon, freq="W-SUN")
        dates = train_dates.append(future_dates)

        # Check reservoir observations for training and evaluation
        if data["reservoir_fill_ratio"].reindex(dates).isna().any():
            return False

        # Check lagged exogenous variables
        if exog_columns:
            external_dates = dates - pd.Timedelta(weeks=horizon)

            if data[exog_columns].reindex(external_dates).isna().any().any():
                return False
    return True

# Check coverage for all data combinations before training
missing_data = []

for group_name, group in search_groups.items():
    for training_weeks, horizon, (label, columns) in product(
        group["training_lengths"], horizons, group["options"].items()):
        if not has_complete_data(weekly_data, group["cutoffs"],
                                 training_weeks, horizon, columns):
            missing_data.append({"group": group_name,
                                 "training_weeks": training_weeks,
                                 "horizon": horizon,
                                 "variables": label})
if missing_data:
    display(pd.DataFrame(missing_data))
else:
    print("All data configurations have complete coverage.")

All data configurations have complete coverage.


#### 3.2.2 Running the grid search

For each configuration, I train a new model at each cutoff and compare the predictions with the actual reservoir filling values. I calculate MAE, RMSE and $R^2$ across the forecast periods and record the computation time. I express MAE and RMSE in percentage points to make the forecast errors easier to interpret. I also test each configuration with and without calendar features representing the annual cycle.

In [12]:
print(inspect.getsource(evaluate_forecast))

def evaluate_forecast(actual, predicted):
    """Calculate MAE, RMSE and R^2 for reservoir filling forecasts."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    if actual.ndim != 1 or predicted.ndim != 1 or len(actual) != len(predicted) or len(actual) == 0:
        raise ValueError("Actual and predicted values must be non-empty, equally sized 1D arrays.")
    if not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError("Actual and predicted values must be finite.")

    # Express forecast errors in percentage points
    mae = mean_absolute_error(actual, predicted) * 100
    rmse = np.sqrt(mean_squared_error(actual, predicted)) * 100

    # R^2 is undefined for a single observation or constant actual values
    r2 = r2_score(actual, predicted) if len(actual) > 1 and np.ptp(actual) > 0 else np.nan

    return {"MAE (pp)": mae, "RMSE (pp)": rmse, "R²": r2}



In [13]:
search_rows = []
search_failures = []
run_id = 0

# Do not start the search if the coverage check found missing data
if missing_data:
    raise ValueError("Some data configurations have incomplete coverage.")

search_started = perf_counter()

for group_name, group in search_groups.items():
    cutoffs = group["cutoffs"]
    group_started = perf_counter()

    print(f"Running {group_name}...")

    for forecast_weeks, training_weeks, (label, columns), calendar in product(
        horizons, group["training_lengths"], group["options"].items(), [False, True]):

        # Collect actual reservoir values for all forecast periods
        actual = np.concatenate([weekly_data["reservoir_fill_ratio"].reindex(
            pd.date_range(start=cutoff + pd.Timedelta(weeks=1), periods=forecast_weeks, freq="W-SUN")).to_numpy()
                                 for cutoff in cutoffs])

        # Test each combination of SNARIMAX parameters
        for p, d, q, m, sp, sd, sq in orders:
            params = {"p": p, "d": d, "q": q, "m": m, "sp": sp, "sd": sd, "sq": sq}
            settings = {"params": params,
                        "training_weeks": training_weeks,
                        "horizon": forecast_weeks,
                        "columns": list(columns),
                        "use_calendar": calendar,
                        "exog_lag": forecast_weeks}
            run_id += 1
            started = perf_counter()

            try:
                # Train a new model and make a forecast at each cutoff
                forecasts = [forecast_reservoir(weekly_data, cutoff=cutoff, **settings) for cutoff in cutoffs]

                # Combine predictions across all forecast periods
                predicted = np.concatenate([forecast["forecast_fill_ratio"].to_numpy() for forecast in forecasts])

                scores = evaluate_forecast(actual, predicted)
                elapsed = perf_counter() - started

                # Store the model settings, evaluation metrics and runtime
                search_rows.append({"run_id": run_id,
                                    "group": group_name,
                                    "horizon": forecast_weeks,
                                    "training_weeks": training_weeks,
                                    "variables": label,
                                    "calendar": calendar,
                                    **params, **scores,
                                    "cutoffs": len(cutoffs),
                                    "seconds": elapsed})

            except Exception as exc:
                search_failures.append({"run_id": run_id,
                                        "group": group_name,
                                        "horizon": forecast_weeks,
                                        "training_weeks": training_weeks,
                                        "variables": label,
                                        "calendar": calendar,
                                        **params,
                                        "error": f"{type(exc).__name__}: {exc}"})

    print(f"Finished {group_name} in {(perf_counter() - group_started) / 60:.2f} minutes.")

search_results = pd.DataFrame(search_rows)
failed_results = pd.DataFrame(search_failures)

print(f"\nSuccessful configurations: {len(search_results):,}")
print(f"Failed configurations: {len(failed_results):,}")
print(f"Total search time: {(perf_counter() - search_started) / 60:.2f} minutes")

if not failed_results.empty:
    display(failed_results.head())

Running long_history...
Finished long_history in 33.31 minutes.
Running medium_history...
Finished medium_history in 3.43 minutes.
Running short_history...
Finished short_history in 1.71 minutes.

Successful configurations: 25,344
Failed configurations: 0
Total search time: 38.45 minutes


#### 3.2.3 Comparing configurations

I first use two simple forecasts as baselines. The naive forecast uses the last observed reservoir filling for all predicted weeks, while the seasonal naive forecast uses the values from 52 weeks earlier. I evaluate both using the same cutoffs and horizons as in the grid search.

In [14]:
reservoir = weekly_data["reservoir_fill_ratio"]
baseline_rows = []

for group_name, group in search_groups.items():
    for horizon in horizons:
        actual = []
        naive = []
        seasonal_naive = []

        for cutoff in group["cutoffs"]:
            future_dates = pd.date_range(start=cutoff + pd.Timedelta(weeks=1),
                                         periods=horizon, freq="W-SUN")

            actual.extend(reservoir.reindex(future_dates).to_numpy())

            # Repeat the last observed reservoir value
            naive.extend([reservoir.loc[cutoff]] * horizon)

            # Use reservoir values from 52 weeks earlier
            seasonal_dates = future_dates - pd.Timedelta(weeks=52)
            seasonal_naive.extend(reservoir.reindex(seasonal_dates).to_numpy())

        # Evaluate both baselines across all forecast periods
        for label, predicted in [("Naive", naive),
                                 ("Seasonal naive", seasonal_naive)]:
            scores = evaluate_forecast(actual, predicted)

            baseline_rows.append({"group": group_name,
                                  "horizon": horizon,
                                  "baseline": label,
                                  **scores})

baseline_results = pd.DataFrame(baseline_rows)

I now compare the configuration with the lowest MAE for each group and horizon with the two baselines. Since the groups have different forecast cutoffs, I compare the results within each group rather than ranking the groups against each other.

In [15]:
# Select the configuration with the lowest MAE for each group and horizon
best_by_group = (search_results.sort_values("MAE (pp)").groupby(["group", "horizon"], sort=False).head(1).sort_values(["group", "horizon"]))

# Compare the best SNARIMAX configurations with both baselines
metrics = ["MAE (pp)", "RMSE (pp)", "R²"]

best_models = best_by_group.copy()
best_models["model"] = "SNARIMAX - " + best_models["variables"]

comparison_columns = ["group", "horizon", "model"] + metrics
comparison = pd.concat([best_models[comparison_columns], baseline_results.rename(columns={"baseline": "model"})[comparison_columns]], ignore_index=True)
comparison = comparison.sort_values(["group", "horizon", "MAE (pp)"])

display(comparison.reset_index(drop=True).round(3))

,group,horizon,model,MAE (pp),RMSE (pp),R²
0,long_history,4,SNARIMAX - Reservoir only,2.506,3.612,0.977
1,long_history,4,Naive,4.889,6.183,0.933
2,long_history,4,Seasonal naive,11.353,14.773,0.619
3,long_history,8,SNARIMAX - Reservoir only,4.336,5.794,0.940
4,long_history,8,Naive,7.903,10.713,0.795
5,long_history,8,Seasonal naive,11.474,14.760,0.610
6,long_history,12,SNARIMAX - EUR/NOK,5.181,6.986,0.904
7,long_history,12,Seasonal naive,11.311,14.591,0.579
8,long_history,12,Naive,12.142,16.641,0.453
9,medium_history,4,SNARIMAX - EUR/NOK,1.495,1.989,0.988


The best SNARIMAX configurations give lower MAE and RMSE than both baselines for all tested horizons in each group. The naive forecast has a lower MAE than the seasonal naive forecast for shorter horizons, but this is not always the case for the 12-week horizon.

The best configurations in the long-history group use only reservoir data for the four- and eight-week horizons, while EUR/NOK is included in the best configurations for all three horizons in the medium-history group. Since these configurations were selected based on the grid search results, I will also evaluate some of them on forecast periods later in 2026 that were not used in the grid search.

#### 3.2.4 Combining exogenous variables

So far, I have tested variables from each data source separately. I now want to see whether combining variables from different sources improves the predictions. Instead of repeating the entire grid search, I reuse the best parameter settings found for each variable selection, training length and horizon. I keep the same forecast cutoffs as before.

In [16]:
def make_combined_options(*sources):
    """Create variable combinations from at least two different sources."""
    options = {}

    for selection in product(*[{"": [], **source}.items() for source in sources]):
        selected = [(label, columns) for label, columns in selection if columns]

        if len(selected) >= 2:
            label = " + ".join(label for label, _ in selected)
            columns = [col for _, cols in selected for col in cols]

            options[label] = columns

    return options

combined_options = {
    "medium_history": make_combined_options(currency_options, weather_options),
    "short_history": make_combined_options(currency_options, weather_options, electricity_options)}

for group_name, options in combined_options.items():
    print(f"{group_name}: {len(options)} additional variable selections")

medium_history: 9 additional variable selections
short_history: 69 additional variable selections


In [17]:
# Select the best configuration for each variable selection
best_by_variables = (search_results.sort_values("MAE (pp)").groupby(["group", "horizon", "training_weeks", "variables"], sort=False).head(1))

param_names = ["p", "d", "q", "m", "sp", "sd", "sq"]

combined_rows = []
combined_failures = []

started = perf_counter()

for (group_name, training_weeks, horizon), candidates in best_by_variables.groupby(["group", "training_weeks", "horizon"]):
    if group_name not in combined_options:
        continue

    cutoffs = search_groups[group_name]["cutoffs"]

    actual = np.concatenate([reservoir.reindex(pd.date_range(start=cutoff + pd.Timedelta(weeks=1), periods=horizon, freq="W-SUN")).to_numpy()
                             for cutoff in cutoffs])

    candidate_settings = candidates[param_names + ["calendar"]].drop_duplicates()

    for row in candidate_settings.to_dict("records"):
        params = {name: int(row[name]) for name in param_names}
        calendar = bool(row["calendar"])

        for label, columns in combined_options[group_name].items():
            settings = {"params": params,
                        "training_weeks": int(training_weeks),
                        "horizon": int(horizon),
                        "columns": columns,
                        "use_calendar": calendar,
                        "exog_lag": int(horizon)}
            
            result = {"group": group_name,
                      "horizon": horizon,
                      "training_weeks": training_weeks,
                      "variables": label,
                      "calendar": calendar,
                      **params}
            
            configuration_started = perf_counter()

            try:
                forecasts = [forecast_reservoir(weekly_data, cutoff=cutoff, **settings) for cutoff in cutoffs]
                predicted = np.concatenate([forecast["forecast_fill_ratio"].to_numpy() for forecast in forecasts])
                scores = evaluate_forecast(actual, predicted)
                combined_rows.append({**result, **scores, "cutoffs": len(cutoffs), "seconds": perf_counter() - configuration_started})

            except Exception as exc:
                combined_failures.append({**result, "error": str(exc)})

combined_results = pd.DataFrame(combined_rows)

print(f"Successful configurations: {len(combined_results)}")
print(f"Failed configurations: {len(combined_failures)}")
print(f"Total time: {(perf_counter() - started):.2f} seconds")

Successful configurations: 1563
Failed configurations: 0
Total time: 28.15 seconds


I compare the best configurations without external data, with data from one external source and with data from multiple sources. For example, temperature and precipitation belong to the same source, while EUR/NOK and precipitation belong to different sources.

In [18]:
combined_comparison_rows = []

for (group_name, training_weeks, horizon), candidates in combined_results.groupby(["group", "training_weeks", "horizon"]):
    original = best_by_variables.loc[
        (best_by_variables["group"] == group_name) & (best_by_variables["training_weeks"] == training_weeks) & (best_by_variables["horizon"] == horizon)]

    best_reservoir = original.loc[original["variables"] == "Reservoir only"].iloc[0]
    best_single = original.loc[original["variables"] != "Reservoir only"].nsmallest(1, "MAE (pp)").iloc[0]
    best_combined = candidates.nsmallest(1, "MAE (pp)").iloc[0]

    for label, best in [("Reservoir only", best_reservoir),
                        ("Single source", best_single),
                        ("Combined sources", best_combined)]:

        combined_comparison_rows.append({"group": group_name,
                                         "training_weeks": training_weeks,
                                         "horizon": horizon,
                                         "source selection": label,
                                         "variables": best["variables"],
                                         **{metric: best[metric] for metric in metrics}})

combined_comparison = pd.DataFrame(combined_comparison_rows)

In [19]:
training_labels = {104: "Medium history (104 weeks)",
                   208: "Medium history (208 weeks)",
                   78: "Short history (78 weeks)"}

plot_data = combined_comparison.copy()
plot_data["training period"] = plot_data["training_weeks"].map(training_labels)

fig = px.bar(plot_data,
             x="horizon",
             y="MAE (pp)",
             color="source selection",
             color_discrete_map={"Reservoir only": "#636EFA",
                                 "Single source": "#EF553B",
                                 "Combined sources": "#00CC96"},
             barmode="group",
             facet_col="training period",
             facet_col_wrap=1,
             facet_row_spacing=0.08,
             hover_data=["variables", "RMSE (pp)", "R²"],
             category_orders={"training period": list(training_labels.values()),
                              "source selection": ["Reservoir only", "Single source", "Combined sources"],
                              "horizon": [4, 8, 12]},
             labels={"horizon": "Horizon (weeks)", "source selection": "External data sources"},
             title="Lowest MAE by data source selection",
             height=850)

fig.update_xaxes(type="category", showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

Combining variables from different sources does not always improve the predictions. In the medium-history group, the differences are generally small for the shorter horizons. However, combining EUR/NOK and temperature improves all three evaluation metrics for the 12-week horizon with 208 weeks of training.

In the short-history group, combining sources gives a lower MAE for all three horizons, although the best single-source model has a lower RMSE for the four-week horizon. I only tested these configurations on four forecast cutoffs in late 2025, and some of the longer forecast periods overlap. I therefore don't know if they would perform equally well at other times of the year.

#### 3.2.5 Evaluation on later dates

Since the four-week forecasts gave the lowest errors in each group, I decided to focus on this horizon when choosing the default settings for Streamlit.

I picked five of the best configurations from the long- and medium-history groups and tested them again using the same forecast dates in 2026. I kept the model settings unchanged. I left out the short-history group, mainly because the electricity data are only available for 2024–2025.

In [20]:
candidates = search_results.loc[(search_results["horizon"] == 4) & (search_results["group"].isin(["long_history", "medium_history"]))].copy()
candidates["source selection"] = np.where(candidates["variables"] == "Reservoir only", "Reservoir only", "Single source")

medium_combined = combined_results.loc[(combined_results["group"] == "medium_history") & (combined_results["horizon"] == 4)].copy()
medium_combined["source selection"] = "Combined sources"

candidates = pd.concat([candidates, medium_combined], ignore_index=True)
selected_configs = (candidates.sort_values("MAE (pp)").groupby(["group", "source selection"], sort=False).head(1).sort_values(["group", "source selection"]))

In [21]:
settings_table = selected_configs.copy()
settings_table["(p, d, q)"] = list(zip(settings_table["p"], settings_table["d"], settings_table["q"]))
settings_table["(sp, sd, sq, m)"] = list(zip(settings_table["sp"], settings_table["sd"], settings_table["sq"], settings_table["m"]))

display(settings_table[["group", "training_weeks", "variables", "calendar", "(p, d, q)", "(sp, sd, sq, m)", "MAE (pp)", "RMSE (pp)", "R²"]].reset_index(drop=True).round(3))

,group,training_weeks,variables,calendar,"(p, d, q)","(sp, sd, sq, m)",MAE (pp),RMSE (pp),R²
0,long_history,780,Reservoir only,True,"(0, 1, 0)","(1, 1, 1, 52)",2.506,3.612,0.977
1,long_history,780,EUR/NOK,False,"(1, 0, 0)","(1, 1, 1, 52)",2.605,3.749,0.975
2,medium_history,208,EUR/NOK + Precipitation,False,"(1, 1, 1)","(0, 1, 1, 52)",1.510,2.176,0.986
3,medium_history,208,Reservoir only,False,"(4, 1, 0)","(0, 1, 0, 52)",1.763,2.487,0.981
4,medium_history,208,EUR/NOK,False,"(4, 1, 0)","(0, 1, 0, 52)",1.495,1.989,0.988


All five configurations use seasonal differencing with a seasonal period of 52 weeks. In the medium-history group, the best reservoir-only and EUR/NOK models differ only in whether the exchange rate is included.

I use eight forecast cutoffs from April to October 2026, spaced four weeks apart so that the forecast periods do not overlap. I train a new model at each cutoff using the selected settings and compare the predictions with the actual reservoir filling values and the two baselines.

In [22]:
test_cutoffs = pd.date_range("2026-04-05", "2026-10-18", freq="4W-SUN")
horizon = 4
test_rows = []

actual = np.concatenate([reservoir.reindex(pd.date_range(start=cutoff + pd.Timedelta(weeks=1), periods=horizon, freq="W-SUN")).to_numpy()
                         for cutoff in test_cutoffs])

for row in selected_configs.to_dict("records"):
    group_name = row["group"]
    options = {**search_groups[group_name]["options"],
               **combined_options.get(group_name, {})}

    columns = options[row["variables"]]
    params = {name: int(row[name]) for name in param_names}
    training_weeks = int(row["training_weeks"])

    if not has_complete_data(weekly_data, test_cutoffs, training_weeks, horizon, columns):
        raise ValueError(f"Incomplete data for {group_name}: {row['variables']}")

    started = perf_counter()

    forecasts = [forecast_reservoir(weekly_data,
                                    params=params,
                                    training_weeks=training_weeks,
                                    horizon=horizon,
                                    cutoff=cutoff,
                                    columns=columns,
                                    use_calendar=bool(row["calendar"]),
                                    exog_lag=horizon)
                 for cutoff in test_cutoffs]

    elapsed = perf_counter() - started

    predicted = np.concatenate([forecast["forecast_fill_ratio"].to_numpy()
                                for forecast in forecasts])

    test_rows.append({"group": group_name,
                      "training_weeks": training_weeks,
                      "variables": row["variables"],
                      "calendar": row["calendar"],
                      **params,
                      **evaluate_forecast(actual, predicted),
                      "seconds_per_forecast": elapsed / len(test_cutoffs)})

test_results = pd.DataFrame(test_rows)

In [23]:
naive = np.concatenate([np.repeat(reservoir.loc[cutoff], horizon) for cutoff in test_cutoffs])
seasonal_naive = np.concatenate([reservoir.reindex(pd.date_range(start=cutoff + pd.Timedelta(weeks=1),
                                                                 periods=horizon, freq="W-SUN") - pd.Timedelta(weeks=52)).to_numpy()
                                 for cutoff in test_cutoffs])

test_baselines = pd.DataFrame([{"variables": "Naive", **evaluate_forecast(actual, naive)},
                               {"variables": "Seasonal naive", **evaluate_forecast(actual, seasonal_naive)}])
test_comparison = pd.concat([test_results, test_baselines], ignore_index=True)

display(test_comparison[["group", "training_weeks", "variables", *metrics]].sort_values("MAE (pp)").reset_index(drop=True).round(3))

,group,training_weeks,variables,MAE (pp),RMSE (pp),R²
0,long_history,780.0,Reservoir only,2.486,4.116,0.947
1,medium_history,208.0,Reservoir only,2.602,3.411,0.963
2,long_history,780.0,EUR/NOK,2.835,4.997,0.922
3,medium_history,208.0,EUR/NOK,3.258,4.488,0.937
4,medium_history,208.0,EUR/NOK + Precipitation,3.667,4.891,0.925
5,NaN,NaN,Naive,4.288,6.131,0.882
6,NaN,NaN,Seasonal naive,25.746,26.245,-1.161


The results from 2026 are quite different from what I found in the grid search. The two models without external variables now perform best. Adding `EUR/NOK` to the model with 208 weeks of training actually increases the error, even though it improved the results in the earlier experiments. All five models still perform better than both baselines.

The model with 780 weeks of training gives the lowest MAE, while the one with 208 weeks gives the lowest RMSE. I decided to use the model with 780 weeks of training as the default in Streamlit, since I have used MAE as my main criterion throughout the experiments.

The default settings for Streamlit are:

- Non-seasonal parameters (p, d, q): (0, 1, 0)
- Seasonal parameters (sp, sd, sq, m): (1, 1, 1, 52)
- Training length: 780 weeks
- Horizon: 4 weeks
- Calendar features: enabled
- External variables: none

Although I tested many different settings, I did not test every possible combination. I also only used data from `NO3`, so other electricity areas may benefit from different model settings or external variables. Users will still be able to change these settings in Streamlit.

## 4. Streamlit application

### 4.1 Preparing geographical boundaries for the map

I converted the shapefiles used in Part 3 to GeoJSON for the interactive map. I combined the five NVE regions into the three VASS areas and added NO0 by merging the five electricity bidding zones.

Since the same GeoJSON can also be used for weather sampling, I decided to remove the original shapefiles from the repository. The conversion code is kept below to show how the GeoJSON was created. If the shapefiles are no longer available, the code simply loads the existing GeoJSON.

For weather sampling, I still use EPSG:25833 to generate locations in projected coordinates before converting them to WGS84 for Open-Meteo.

In [24]:
data_path = Path("./data/geographical")

price_path = data_path / "Budområder" / "Budområder.shp"
water_path = data_path / "NVE_Region" / "NVE_Region.shp"
geojson_path = data_path / "reservoir_areas.geojson"

# Generate the GeoJSON if the original shapefiles are available
if price_path.exists() and water_path.exists():
    price_areas = gpd.read_file(price_path, encoding="utf-8")
    water_areas = gpd.read_file(water_path, encoding="utf-8")

    # Match electricity bidding zones to Magasinstatistikk identifiers
    price_areas["area_type"] = "EL"
    price_areas["area_number"] = price_areas["budomr"].str.replace(" ", "", regex=False).str[2:].astype(int)

    # Combine the NVE regions into the three VASS areas
    water_areas["area_type"] = "VASS"
    water_areas["area_number"] = water_areas["REGION"].map({"RS": 1, "RØ": 1, "RV": 2, "RM": 3, "RN": 3})

    water_areas = water_areas.dissolve(by=["area_type", "area_number"], as_index=False)

    # Combine the electricity areas to represent NO0
    national_area = gpd.GeoDataFrame({"area_type": ["NO"], "area_number": [0]},
                                      geometry=[price_areas.geometry.union_all()],
                                      crs=price_areas.crs)

    # Keep only the properties needed for the map
    columns = ["area_type", "area_number", "geometry"]

    areas = gpd.GeoDataFrame(pd.concat([price_areas[columns],
                                       water_areas[columns],
                                       national_area[columns]], ignore_index=True),
                              crs=price_areas.crs)

    areas["area_id"] = areas["area_type"] + areas["area_number"].astype(str)

    # Convert to WGS84 for Folium and GeoJSON
    areas = areas.to_crs(epsg=4326)

    areas.to_file(geojson_path, driver="GeoJSON", index=False)

# Reuse the GeoJSON after the original shapefiles have been removed
elif geojson_path.exists():
    areas = gpd.read_file(geojson_path)

else:
    raise FileNotFoundError("Neither the original shapefiles nor the GeoJSON are available.")

areas

,area_type,area_number,geometry,area_id
0,EL,1,"POLYGON ((11.80905 62.8288, 11.81389 62.8274, ...",EL1
1,EL,2,"POLYGON ((5.74061 60.3517, 5.74081 60.35085, 5...",EL2
2,EL,3,"POLYGON ((10.87378 65.41986, 10.98433 65.40083...",EL3
3,EL,4,"POLYGON ((28.33596 71.29423, 28.36301 71.29201...",EL4
4,EL,5,"POLYGON ((7.41477 61.85483, 7.41637 61.85293, ...",EL5
5,VASS,1,"POLYGON ((10.57616 58.76183, 10.35937 58.77128...",VASS1
6,VASS,2,"POLYGON ((9.06206 62.37261, 9.03652 62.36454, ...",VASS2
7,VASS,3,"POLYGON ((15.51729 66.55129, 15.45106 66.52397...",VASS3
8,NO,0,"POLYGON ((12.11178 62.70625, 12.09709 62.68135...",NO0


In [25]:
print("CRS:", areas.crs)
print("Number of areas:", len(areas))
print("Unique area identifiers:", areas["area_id"].is_unique)
print("All geometries valid:", areas.is_valid.all())
print("Empty geometries:", areas.is_empty.sum())

CRS: EPSG:4326
Number of areas: 9
Unique area identifiers: True
All geometries valid: True
Empty geometries: 0


<div style="background-color:#EAF2F7; border-left:5px solid #3E7FA6; padding:12px 16px; border-radius:6px;">

<h2>AI usage</h2>
To be filled out...
</div>

## Project log

I started by looking into the Norges Bank API and made a function to download exchange rates for different currencies, dates and frequencies. I put the function in a separate module so I can reuse it in Streamlit later, and kept the original metadata since some of it was useful for the timezone investigation. I tested the function with EUR and USD rates over Christmas 2025 and confirmed that there were no observations during the Christmas holidays or the weekend when using Business frequency.

I then looked at how dates and timezones are handled in the four datasets used in the project. For ENTSO-E and Open-Meteo, I reused my work from Parts 2 and 3 to check how the timestamps behave around the daylight-saving change in October 2025. Both examples showed the repeated local hour with the correct UTC offsets, so I didn't need to change the existing timestamp handling. I also realised that using the same timezone would not be enough to combine the datasets since they have different observation intervals. I decided to use weekly data for forecasting, with different aggregation methods depending on the variable.

I prepared the weekly datasets for NO3, using weather data from 100 sampled locations from 2021 onwards. One difficulty was that the datasets cover very different periods, especially the electricity data, which I only collected for 2024–2025. I therefore divided the forecasting experiments into three groups with different training lengths. This allowed me to test the longer reservoir and currency histories without limiting all experiments to the period for which electricity data were available.

I implemented a forecasting function using River's SNARIMAX and used a grid search to compare different settings more systematically. The final grid search took about 38 minutes. I also tested combinations of variables from different sources, reusing the best model parameters instead of repeating the entire search.

Some of the models with external variables looked very promising at first, particularly those using EUR/NOK. However, when I tested five selected configurations on later dates in 2026, the models without external variables performed better. I ended up choosing a model with 780 weeks of training and a four-week horizon as the default for Streamlit, since it gave the lowest MAE in this final comparison and does not require loading any additional datasets.

Before changing the Streamlit app, I planned how to organise the pages and which selections should be shared between them. I decided to keep the region selectors in the sidebar and divide the navigation into Data overview, Data analysis and Forecasting. I also changed to a dark theme, added a currency exchange rates page and merged the two reservoir pages, replacing the LineChartColumn overview with summary metrics.

For the interactive map, I converted the shapefiles to a single GeoJSON file containing all nine Magasinstatistikk areas. I realised that I could also use this file for weather sampling, so I updated the sampling function and removed the original shapefiles to avoid storing the same geographical data twice. I tested the updated function for all nine areas and checked that the existing weather functionality in Streamlit still worked.

I used Folium to create an interactive map on the Home page and connected it to the existing region selectors using session state. Users can select a region either on the map or in the sidebar and colour the areas by reservoir or weather measurements. I used fixed colour scales so that the colours can be compared across different weeks. For the weather data, I reused my existing hourly download and weekly aggregation functions, using 100 sampled locations per region.

Since downloading weather data for all regions can take some time, I decided to load them only when requested by the user and cache the results. I also added a progress bar to show the loading progress. ERA5 data are published with a delay of approximately five days, so weather data may not yet be available for the latest reservoir observation. In this case, the app still displays the reservoir measurements and informs the user about the delay.

I then replaced the Matplotlib plots on the Reservoir Analysis page with interactive Plotly figures, keeping the existing STL and spectrogram calculations. I also changed the STL axes to display percentages and percentage points and used calendar dates instead of relative years in the spectrogram.

I also updated the Weather Analysis page so that both SPC and LOF can be used with temperature or precipitation, with temperature as the default. Since both variables are already downloaded in the same API request, switching between them does not require downloading new data. I kept the existing analysis methods and replaced the Matplotlib plots with Plotly figures, also making the plot labels and tables adapt to the selected weather property.

To be continued...